In [1]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.messages import HumanMessage,AIMessage,SystemMessage

load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [2]:
os.environ['GROQ_API_KEY'] is None

False

In [3]:
slots = {
    "10 AM": {"capacity": 20, "booked": 0},
    "12 PM": {"capacity": 20, "booked": 0},
    "2 PM": {"capacity": 20, "booked": 0}
}

In [4]:
@tool
def check_slots():
    """ a tool to check the available slots for booking a ticket"""
    available_slots={}

    for slot,details in slots.items():
        available_slots[slot]=details['capacity']-details['booked']

    return str(available_slots)

bookings = []
@tool
def book_slot(name:str,people:int,slot:str): #slot=10 am
    """a tool that will actually book the movie slot fro you based on the details we gave"""

    if slot not in slots:
        return "Invalid slot. Choose 10 AM, 12 PM or 2 PM."

    if people<1:
        return "Number of people must be at least 1."
    

    available = slots[slot]['capacity'] - slots[slot]['booked']

    if people>available:
        return f"Sorry!We Only have {available} seats available for the {slot} slot."

    slots[slot]['booked']+=people

    bookings.append({
        "name": name,
        "people": people,
        "slot": slot
    })

    return f"Booking successful for {name}: {people} people at {slot}."

@tool
def show_bookings():
    """Show all bookings and total booked people."""

    total = sum(booking["people"] for booking in bookings)

    return f"Bookings: {bookings}\nTotal people booked: {total}"


In [5]:
llm=ChatGroq(model='openai/gpt-oss-120b')


In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent=create_agent(
    model=llm,
    tools=[check_slots,book_slot,show_bookings],
    system_prompt="""
    You are a friendly event ticket booking assistant.
    1.ask user name, group of people and slot they want.
    2.use the check_slots toool to check the available slots.
    3.if not available slot then deal with customer and meniton there that sorry we dont have available slots.clear
    4.before booking make sure you have all the details that required.
    5.also use the book_slots for booking.
    6. Never claim a booking succeeded unless the tool confirms it.
    7. Use show_bookings when asked to see all bookings.
    8. Never exceed the available capacity.
    """,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_slot":{
                    "allowed_decisions":['approve','reject']
                },
                "show_bookings":False
            }
        )
    ]
)

In [7]:
print("Agent created successfully!")

Agent created successfully!


In [8]:

from langgraph.types import Command

config = {"configurable": {"thread_id": "chat4"}}

while True:
    user_input = input("YOU: ").strip()

    if user_input.lower() == "exit":
        print("BOT: Goodbye!")
        break

    # First message or continue the existing conversation
    if "__interrupt__" not in result if "result" in globals() else True:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": user_input}]},
            config=config
        )

    # Handle booking approval
    while "__interrupt__" in result:
        while True:
            user_input = input("YOU (yes/no): ").lower().strip()

            if user_input in ["yes", "y", "approve"]:
                decision = {"type": "approve"}
                break

            elif user_input in ["no", "n", "reject"]:
                decision = {
                    "type": "reject",
                    "message": "User rejected the booking"
                }
                break

            else:
                print("Please enter yes or no.")

        result = agent.invoke(
            Command(resume={"decisions": [decision]}),
            config=config
        )

    print("\nBOT:", result["messages"][-1].content)



BOT: Sure! I'd be happy to help you book tickets.

Could you please tell me:

1. Your name  
2. How many people are in your group  
3. Which time slot you’d like to book (e.g., “6 PM”, “7:30 PM”, etc.)

Once I have those details, I’ll check the availability for you.

BOT: Welcome! 😊

Could you let me know:

1. **Your name**  
2. **How many people** are in your group  
3. **Which time slot** you’d like to book (e.g., “6 PM”, “7:30 PM”, “8 PM”, etc.)

Once I have those details, I’ll check the availability for you.

BOT: Great! There are currently no bookings.

Whenever you’re ready, just let me know:

1. Your name  
2. How many people are in your group  
3. Which time slot you’d like (e.g., “6 PM”, “7:30 PM”, etc.)

I’ll check availability and secure the tickets for you!

BOT: Here’s the current availability:

- **10 AM** – 20 seats left  
- **12 PM** – 20 seats left  
- **2 PM** – 20 seats left  

Let me know which slot you’d like to book, along with:

1. Your name  
2. The number of p